# Aprendizado de Autômatos vs Aprendizado de Máquina
## Uma Comparação Experimental para Model-Based Testing

Este notebook implementa e compara duas estratégias de inferência automática de modelos comportamentais a partir de logs de execução real, no contexto de **Model-Based Testing (MBT)**.

### Estratégias comparadas

**Aprendizado de Autômatos (AALpy)**
Produz modelos formais — Máquinas de Estados Finitos (MEF) — através de algoritmos passivos (RPNI, GSM) e ativo (L*). Modelos formais são verificáveis, interpretáveis e generalizáveis além dos dados observados.

**Aprendizado Estatístico (Agilkia)**
Trabalha diretamente com os traces sem inferir um modelo formal. Captura comportamentos observados nos logs com alta fidelidade aos dados reais.

### Dimensões de avaliação

- **Cobertura:** quanto do espaço de comportamentos possíveis foi exercitado
- **Mutation Score:** capacidade de detectar falhas injetadas propositalmente

### Dataset
Logs reais de um sistema de scanner de supermercado, disponível no repositório oficial do Agilkia (PHILAE-PROJECT). Contém 65.273 eventos organizados em 4.818 sessões de clientes.

## 1. Instalação e Dependências

In [ ]:
!pip install aalpy agilkia -q

import pandas as pd
import numpy as np
import re
import random
import time
import matplotlib.pyplot as plt
from IPython.display import Image
import os

print('Dependências instaladas com sucesso.')

## 2. Carregamento e Pré-processamento dos Dados

Os logs brutos passam por dois processos de semântização antes de serem usados:

**Semântização dos parâmetros:** códigos numéricos dos produtos são convertidos em rótulos semânticos. Valores acima de 100.000 são identificados como códigos de produto (`produto`); valores menores como quantidades ou valores monetários (`valor/qtd`); referências a caixa registradora (`caixa`).

**Semântização do status:** os códigos de retorno do sistema são mapeados para rótulos legíveis:
- `0` → `ok` (operação bem sucedida)
- `-2` → `erro` (falha na operação)
- `?` → `void` (operação sem retorno)
- valores decimais → `valor` (retorno monetário)

O resultado é um alfabeto de **9 eventos únicos** e **4.818 traces** de sessão, cada um representando o comportamento de um cliente.

In [ ]:
from google.colab import files

print('Faça o upload do arquivo 127.0.0.1-1571403244552.csv')
uploaded = files.upload()

filename = '127.0.0.1-1571403244552.csv'
df = pd.read_csv(filename)
df.columns = ['id', 'timestamp', 'client', 'scan', 'action', 'params', 'status']
df = df.drop(columns=['timestamp'])

print(f'DataFrame carregado. Linhas: {len(df)}')
df.head()

In [ ]:
def parse_params_raw(x):
    if not isinstance(x, str): return None
    x = x.strip()
    if x in ('[]', '[ ]', ''): return None
    match = re.findall(r'\[([^\]]*)\]', x)
    if match:
        return match[0].strip().replace("'", '')
    return None

def simplify_params_semantic(action, param_raw):
    if param_raw is None: return ''
    param_raw_lower = param_raw.lower()
    if 'caisse' in param_raw_lower: return '(caixa)'
    try:
        val = float(param_raw)
        return '(produto)' if val > 100000 else '(valor/qtd)'
    except ValueError:
        return f'({param_raw})' if len(param_raw) < 15 else '(param_longo)'

def simplificar_status(s):
    s = str(s).strip()
    if s == '0':    return 'ok'
    elif s == '-2': return 'erro'
    elif s == '?':  return 'void'
    else:
        try:
            float(s)
            return 'valor'
        except:
            return 'outro'

df['param_raw']  = df['params'].apply(parse_params_raw)
df['event']      = df.apply(lambda row: f"{row['action']}{simplify_params_semantic(row['action'], row['param_raw'])}", axis=1)
df['status_sem'] = df['status'].apply(simplificar_status)

traces = df.groupby('client')['event'].apply(list).tolist()
traces_com_status = (
    df.groupby('client')
      .apply(lambda g: list(zip(g['event'], g['status_sem'])), include_groups=False)
      .tolist()
)
alphabet_real = list(df['event'].unique())

print(f'Alfabeto: {len(alphabet_real)} eventos únicos')
print(f'Alfabeto: {alphabet_real}')
print(f'Traces gerados: {len(traces)}')
print(f'\nDistribuição dos status:')
print(df['status_sem'].value_counts())

## 3. Aprendizado Passivo — RPNI e GSM

Algoritmos passivos recebem apenas traces positivos (sequências que o sistema executou) e tentam inferir um autômato que os aceite. Não há interação com o sistema — apenas os dados históricos são usados.

**RPNI (Regular Positive and Negative Inference):** constrói uma PTA (Prefix Tree Acceptor) a partir dos traces positivos e realiza merges de estados compatíveis.

**GSM (General State Merging):** variante mais flexível do RPNI, com comportamento de saída no estilo Moore machine.

**Resultado esperado:** sem traces negativos (exemplos do que o sistema *não faz*), ambos os algoritmos colapsam em **1 estado** — um modelo degenerado que aceita qualquer sequência e não possui poder discriminatório. Isso evidencia a limitação do aprendizado passivo sem contraexemplos para aplicações de MBT.

In [ ]:
from aalpy.learning_algs.deterministic_passive.ClassicRPNI import ClassicRPNI
from aalpy.utils import visualize_automaton

positives = [(t, True) for t in traces]

print('Executando RPNI...')
rpni_helper = ClassicRPNI(positives, 'dfa', print_info=True)
dfa_rpni = rpni_helper.run_rpni()
print(f'RPNI: autômato com {len(dfa_rpni.states)} estado(s).')

visualize_automaton(dfa_rpni, path='dfa_rpni', file_type='png')
time.sleep(2)
if os.path.exists('dfa_rpni.png'):
    display(Image('dfa_rpni.png'))
else:
    print('Imagem ainda não disponível. Execute a célula novamente.')

In [ ]:
from aalpy.learning_algs import run_GSM

gsm_data = [(list(map(str, t)), True) for t in traces]

print('Executando GSM...')
dfa_gsm = run_GSM(gsm_data, output_behavior='moore', transition_behavior='deterministic')
print(f'GSM: autômato com {len(dfa_gsm.states)} estado(s).')

visualize_automaton(dfa_gsm, path='dfa_gsm', file_type='png')
time.sleep(2)
display(Image('dfa_gsm.png'))

## 4. Aprendizado Ativo — L* com SUL baseado em Trie

O algoritmo L* (Angluin, 1987) aprende um autômato interagindo com o sistema através de dois tipos de consulta:

- **Membership Query (MQ):** pergunta ao sistema qual a saída para uma sequência específica de entradas
- **Equivalence Query (EQ):** verifica se a hipótese atual representa corretamente o sistema; se não, retorna um contraexemplo

### SUL baseado em Trie

Como não temos acesso ao sistema real em execução, o SUL (System Under Learning) é construído a partir dos logs reais via estrutura **Trie** — uma árvore onde cada caminho representa uma sequência de eventos observada, e cada nó armazena a saída real registrada nos logs.

Dado qualquer sequência de eventos, o SUL percorre a Trie e retorna a saída observada. Se o caminho nunca foi observado nos logs, retorna `nao_observado` — indicando comportamento fora do escopo dos dados disponíveis.

Essa abordagem garante que o modelo aprenda apenas comportamentos reais, sem inferências além do que os logs mostram.

In [ ]:
from aalpy.SULs import SUL

class LogSUL(SUL):
    """
    SUL construído a partir dos traces reais do CSV via estrutura Trie.
    Dado qualquer sequência de eventos, consulta o caminho na Trie
    e retorna a saída observada nos logs reais.
    Se o caminho não foi observado, retorna 'nao_observado'.
    """
    def __init__(self, traces_com_status):
        super().__init__()
        self.trie = self._construir_trie(traces_com_status)
        self.no_atual = self.trie

    def _construir_trie(self, traces_com_status):
        trie = {}
        for trace in traces_com_status:
            no_atual = trie
            for evento, status in trace:
                evento = evento.strip()
                if evento not in no_atual:
                    no_atual[evento] = {'__saida__': status, '__filhos__': {}}
                no_atual = no_atual[evento]['__filhos__']
        return trie

    def pre(self):
        self.no_atual = self.trie

    def post(self):
        pass

    def step(self, letter):
        letter = letter.strip()
        if letter in self.no_atual:
            saida = self.no_atual[letter]['__saida__']
            self.no_atual = self.no_atual[letter]['__filhos__']
        else:
            saida = 'nao_observado'
            self.no_atual = {}
        return saida

print('LogSUL (Trie) definido.')

In [ ]:
from aalpy.learning_algs import run_Lstar
from aalpy.oracles import RandomWpMethodEqOracle

sul_trie    = LogSUL(traces_com_status)
oracle_trie = RandomWpMethodEqOracle(
    alphabet_real, sul_trie,
    min_length=2, expected_length=15, num_tests=2000
)

model_trie = run_Lstar(
    alphabet_real, sul_trie, oracle_trie,
    automaton_type='mealy'
)

print(f'\nModelo aprendido com {len(model_trie.states)} estados.')
visualize_automaton(model_trie, path='model_trie', file_type='png')
time.sleep(2)
display(Image('model_trie.png'))

In [ ]:
print('Transições do modelo aprendido:')
for state in model_trie.states:
    for symbol in alphabet_real:
        destino = state.transitions[symbol].state_id
        saida   = state.output_fun[symbol]
        print(f'  {state.state_id} --[{symbol.strip()}]--> {destino}  (saída: {saida})')

## 5. Cobertura de Transições do Modelo — AALpy

A cobertura de transições é uma métrica clássica de Model-Based Testing que mede o percentual de transições do modelo exercitadas pelos traces reais:

$$\text{Cobertura} = \frac{\text{transições visitadas}}{\text{total de transições do modelo}}$$

Uma **transição** é a combinação `(estado_origem, evento)` — ou seja, a partir de qual estado e com qual evento a transição é disparada, independente do destino. Com 4 estados e 9 eventos, o modelo possui 36 transições possíveis.

As transições não cobertas correspondem a sequências que **nunca ocorreram** nos logs reais — a maioria leva ao estado lixo (s3), que representa comportamentos inválidos no sistema.

In [ ]:
state_map_trie = {s.state_id: s for s in model_trie.states}

todas_transicoes = set()
for state in model_trie.states:
    for symbol in alphabet_real:
        todas_transicoes.add((state.state_id, symbol))

transicoes_cobertas = set()
for trace in traces_com_status:
    estado_atual = model_trie.initial_state
    for evento, status in trace:
        evento_strip = evento.strip()
        simbolo = next((a for a in alphabet_real if a.strip() == evento_strip), None)
        if simbolo is None: break
        transicoes_cobertas.add((estado_atual.state_id, simbolo))
        estado_atual = estado_atual.transitions[simbolo]

transicoes_nao_cobertas = todas_transicoes - transicoes_cobertas
cobertura_pct = len(transicoes_cobertas) / len(todas_transicoes) * 100

print('=' * 50)
print(' COBERTURA DE TRANSIÇÕES — AALpy (L*)')
print('=' * 50)
print(f'Estados no modelo    : {len(model_trie.states)}')
print(f'Transições totais    : {len(todas_transicoes)}')
print(f'Transições cobertas  : {len(transicoes_cobertas)}')
print(f'Cobertura            : {cobertura_pct:.1f}%')
print()
print('Transições COBERTAS:')
for (estado, simbolo) in sorted(transicoes_cobertas):
    s = state_map_trie[estado]
    destino = s.transitions[simbolo].state_id
    saida   = s.output_fun[simbolo]
    print(f'  ✔ {estado} --[{simbolo.strip()}]--> {destino}  (saída: {saida})')
print()
print('Transições NÃO COBERTAS (sequências nunca observadas nos logs):')
for (estado, simbolo) in sorted(transicoes_nao_cobertas):
    s = state_map_trie[estado]
    destino = s.transitions[simbolo].state_id
    saida   = s.output_fun[simbolo]
    print(f'  ✘ {estado} --[{simbolo.strip()}]--> {destino}  (saída: {saida})')
print('=' * 50)

## 6. Agilkia — Cobertura Comportamental

O Agilkia é um framework de AI-for-Testing que trabalha diretamente com os traces sem inferir um modelo formal. Os traces são carregados em um `TraceSet` — estrutura nativa do Agilkia — e a cobertura é medida por duas métricas:

- **EventCoverage (Action+Status):** mede a proporção de combinações `(action, status)` observadas no subconjunto selecionado sobre o total observado em todos os traces
- **EventPairCoverage:** mede a proporção de pares de eventos consecutivos `(evento_i, evento_i+1)` cobertos

A análise de curva de cobertura responde a pergunta: **com quantos traces já se atinge cobertura suficiente?** Isso é relevante para otimização do esforço de teste — identificar o ponto de saturação onde traces adicionais não contribuem com novos comportamentos.

In [ ]:
import agilkia

trace_list = []
for client_id, group in df.groupby('client'):
    events = []
    for _, row in group.iterrows():
        event = agilkia.Event(
            action=row['event'].strip(),
            inputs={'params': str(row['params'])},
            outputs={'status': simplificar_status(row['status'])}
        )
        events.append(event)
    trace_list.append(agilkia.Trace(events, meta_data={'client': client_id}))

trace_set = agilkia.TraceSet(trace_list)
print(f'TraceSet criado com {len(trace_set)} traces.')
print(f'Actions únicas: {set(ev.action for t in trace_set for ev in t)}')

In [ ]:
asc = agilkia.EventCoverage(event_to_str=lambda ev: f"{ev.action}_{ev.outputs.get('status','')}")
epc = agilkia.EventPairCoverage(event_to_str=lambda ev: ev.action)

tamanhos = [10, 50, 100, 200, 500, 1000, 2000, 4818]
resultados_cob = []

for n in tamanhos:
    solution = np.zeros(len(trace_set))
    solution[:n] = 1
    asc.set_data(trace_set, select=n)
    epc.set_data(trace_set, select=n)
    cob_asc = asc.evaluate(solution)
    cob_epc = epc.evaluate(solution)
    resultados_cob.append((n, cob_asc, cob_epc))
    print(f'N={n:5d} | Action+Status: {cob_asc:.1%} | EventPair: {cob_epc:.1%}')

## 7. Teste de Mutação

O teste de mutação avalia a **capacidade de detecção de falhas** de um modelo. Defeitos são injetados propositalmente no sistema (mutantes) e verifica-se se o modelo consegue detectá-los:

$$\text{Mutation Score} = \frac{\text{mutantes detectados}}{\text{total de mutantes}}$$

### Operadores implementados

Os mutantes foram gerados **manualmente em Python** diretamente sobre os traces de log. Ferramentas como mutpy atuam sobre código fonte Python — não sobre dados de log. Para mutação de dados em MBT não existe ferramenta consolidada na literatura, sendo a implementação manual a abordagem adotada nos artigos de referência.

| Operador | Tipo Formal | Descrição | Taxa |
|----------|-------------|-----------|------|
| `mutar_output` | Output Fault | Troca o status de eventos aleatoriamente | 5% |
| `mutar_transicao` | Transfer Fault | Inverte a ordem de eventos consecutivos | 5% |
| `mutar_remover_evento` | Omission Fault | Remove eventos dos traces | 5% |

Reprodutibilidade garantida via `random.seed(42)` — os mesmos mutantes são gerados sempre.

### Método de detecção

**AALpy:** treina um novo modelo com os traces mutados e compara com o modelo original usando 500 sequências aleatórias. Se os modelos produzirem saídas diferentes em alguma sequência → mutante detectado (morto).

**Agilkia:** compara os conjuntos de pares de eventos consecutivos `(evento_i, evento_i+1)` entre os traces originais e mutados. Se algum par aparecer ou desaparecer → mutante detectado.

Os métodos são **diferentes por natureza** — não injustos, mas distintos. O Agilkia é extremamente sensível a qualquer perturbação nos dados; o AALpy aprende um modelo compacto que abstrai pequenas variações.

In [ ]:
random.seed(42)

def mutar_output(traces, taxa=0.05):
    """Output Fault: troca o status de alguns eventos."""
    outputs_possiveis = ['ok', 'erro', 'void', 'valor']
    mutante = []
    for trace in traces:
        novo_trace = []
        for evento, status in trace:
            if random.random() < taxa:
                novo_status = random.choice([s for s in outputs_possiveis if s != status])
                novo_trace.append((evento, novo_status))
            else:
                novo_trace.append((evento, status))
        mutante.append(novo_trace)
    return mutante

def mutar_transicao(traces, taxa=0.05):
    """Transfer Fault: inverte a ordem de eventos consecutivos."""
    mutante = []
    for trace in traces:
        novo_trace = list(trace)
        if len(novo_trace) > 2 and random.random() < taxa:
            i = random.randint(0, len(novo_trace) - 2)
            novo_trace[i], novo_trace[i+1] = novo_trace[i+1], novo_trace[i]
        mutante.append(novo_trace)
    return mutante

def mutar_remover_evento(traces, taxa=0.05):
    """Omission Fault: remove eventos dos traces."""
    mutante = []
    for trace in traces:
        novo_trace = [(e, s) for e, s in trace if random.random() > taxa]
        mutante.append(novo_trace if novo_trace else trace)
    return mutante

operadores = [
    ('output_fault',   mutar_output),
    ('transfer_fault', mutar_transicao),
    ('remove_evento',  mutar_remover_evento),
]
print('Operadores de mutação definidos.')

In [ ]:
def modelos_divergem(model_original, model_mutante, alphabet, n_testes=500):
    """Retorna True se os dois modelos divergem em alguma sequência aleatória."""
    for _ in range(n_testes):
        sequencia = [random.choice(alphabet) for _ in range(random.randint(1, 10))]
        model_original.reset_to_initial()
        model_mutante.reset_to_initial()
        saida_orig = [model_original.step(s) for s in sequencia]
        saida_mut  = [model_mutante.step(s) for s in sequencia]
        if saida_orig != saida_mut:
            return True
    return False

N_MUTANTES = 30
resultados_aalpy = []

print('=' * 50)
print(' MUTATION SCORE — AALpy (L*)')
print('=' * 50)

for nome_op, operador in operadores:
    mortos = 0
    for i in range(N_MUTANTES):
        traces_mutados = operador(traces_com_status)
        sul_mut = LogSUL(traces_mutados)
        oracle_mut = RandomWpMethodEqOracle(
            alphabet_real, sul_mut,
            min_length=2, expected_length=10, num_tests=500
        )
        model_mut = run_Lstar(
            alphabet_real, sul_mut, oracle_mut,
            automaton_type='mealy', print_level=0
        )
        detectado = modelos_divergem(model_trie, model_mut, alphabet_real)
        if detectado:
            mortos += 1
            print(f'  [{nome_op}] mutante {i+1:02d}: MORTO ✓')
        else:
            print(f'  [{nome_op}] mutante {i+1:02d}: SOBREVIVEU ✗')
    score = mortos / N_MUTANTES * 100
    resultados_aalpy.append((nome_op, mortos, N_MUTANTES, score))
    print(f'  → {nome_op}: {mortos}/{N_MUTANTES} = {score:.1f}%\n')

total_mortos_aalpy = sum(m for _, m, _, _ in resultados_aalpy)
total_mut_aalpy    = sum(t for _, _, t, _ in resultados_aalpy)
print(f'TOTAL AALpy: {total_mortos_aalpy}/{total_mut_aalpy} = {total_mortos_aalpy/total_mut_aalpy*100:.1f}%')
print('=' * 50)

In [ ]:
def pares_unicos(traceset):
    """Extrai todos os pares (action_status_i, action_status_i+1) únicos."""
    pares = set()
    for trace in traceset:
        eventos = [f"{ev.action}_{ev.outputs.get('status','')}" for ev in trace]
        for i in range(len(eventos) - 1):
            pares.add((eventos[i], eventos[i+1]))
    return pares

def mutation_score_agilkia(traces_mutados):
    """Detecta mutante comparando pares únicos de eventos entre original e mutado."""
    trace_list_mut = []
    for trace in traces_mutados:
        events = [agilkia.Event(
            action=ev.strip(),
            inputs={},
            outputs={'status': st}
        ) for ev, st in trace]
        trace_list_mut.append(agilkia.Trace(events))
    ts_mutado = agilkia.TraceSet(trace_list_mut)
    pares_orig = pares_unicos(trace_set)
    pares_mut  = pares_unicos(ts_mutado)
    return len(pares_orig.symmetric_difference(pares_mut)) > 0

resultados_agilkia = []

print('=' * 50)
print(' MUTATION SCORE — Agilkia')
print('=' * 50)

for nome_op, operador in operadores:
    mortos = 0
    for i in range(N_MUTANTES):
        traces_mutados = operador(traces_com_status)
        detectado = mutation_score_agilkia(traces_mutados)
        if detectado:
            mortos += 1
            print(f'  [{nome_op}] mutante {i+1:02d}: MORTO ✓')
        else:
            print(f'  [{nome_op}] mutante {i+1:02d}: SOBREVIVEU ✗')
    score = mortos / N_MUTANTES * 100
    resultados_agilkia.append((nome_op, mortos, N_MUTANTES, score))
    print(f'  → {nome_op}: {mortos}/{N_MUTANTES} = {score:.1f}%\n')

total_mortos_ag = sum(m for _, m, _, _ in resultados_agilkia)
total_mut_ag    = sum(t for _, _, t, _ in resultados_agilkia)
print(f'TOTAL Agilkia: {total_mortos_ag}/{total_mut_ag} = {total_mortos_ag/total_mut_ag*100:.1f}%')
print('=' * 50)

In [ ]:
def modelos_divergem_passivo(model_orig, model_mut, n_testes=500):
    """Versão unificada para DFA (RPNI) e Moore machines (GSM)."""
    alfabeto = model_orig.get_input_alphabet()
    for _ in range(n_testes):
        sequencia = [random.choice(alfabeto) for _ in range(random.randint(1, 10))]
        model_orig.reset_to_initial()
        model_mut.reset_to_initial()
        for s in sequencia:
            model_orig.step(s)
            model_mut.step(s)
        try:
            orig_out = model_orig.current_state.is_accepting
            mut_out  = model_mut.current_state.is_accepting
        except AttributeError:
            orig_out = model_orig.current_state.output
            mut_out  = model_mut.current_state.output
        if orig_out != mut_out:
            return True
    return False

def rodar_mutacao_passivo(model_original, nome_modelo, n_mutantes=30):
    print('=' * 50)
    print(f' MUTATION SCORE — {nome_modelo}')
    print('=' * 50)
    resultados = []
    for nome_op, operador in operadores:
        mortos = 0
        for i in range(n_mutantes):
            traces_mutados = operador(traces_com_status)
            positives = [(list(map(str, [e for e, s in t])), True) for t in traces_mutados]
            if nome_modelo == 'RPNI':
                helper = ClassicRPNI(positives, 'dfa', print_info=False)
                model_mut = helper.run_rpni()
            else:
                gsm_data = [(list(map(str, [e for e, s in t])), True) for t in traces_mutados]
                try:
                    model_mut = run_GSM(gsm_data, output_behavior='moore', transition_behavior='deterministic')
                except:
                    print(f'  [{nome_op}] mutante {i+1:02d}: ERRO ✗')
                    continue
            detectado = modelos_divergem_passivo(model_original, model_mut)
            if detectado:
                mortos += 1
                print(f'  [{nome_op}] mutante {i+1:02d}: MORTO ✓')
            else:
                print(f'  [{nome_op}] mutante {i+1:02d}: SOBREVIVEU ✗')
        score = mortos / n_mutantes * 100
        resultados.append((nome_op, mortos, n_mutantes, score))
        print(f'  → {nome_op}: {mortos}/{n_mutantes} = {score:.1f}%\n')
    total_m = sum(m for _, m, _, _ in resultados)
    total_t = sum(t for _, _, t, _ in resultados)
    print(f'TOTAL {nome_modelo}: {total_m}/{total_t} = {total_m/total_t*100:.1f}%')
    print('=' * 50)
    return resultados

resultados_rpni = rodar_mutacao_passivo(dfa_rpni, 'RPNI')
resultados_gsm  = rodar_mutacao_passivo(dfa_gsm,  'GSM')

## 8. Abordagem de Aichernig — Mutação da MEF

Baseada nos artigos de Aichernig & Tappler (2017, 2019), esta abordagem muta o próprio modelo MEF para gerar **sequências distinguidoras** — sequências que produzem saídas diferentes entre o modelo original e o mutante.

Essas sequências são usadas como contraexemplos direcionados no oráculo de equivalência do L*, substituindo a busca aleatória por uma busca focada nas regiões onde o modelo pode estar errado.

### Operadores aplicados sobre a MEF

- **Output Fault:** troca a saída de uma transição específica
- **Transfer Fault:** redireciona uma transição para outro estado

### Por que não se aplica ao Agilkia

A abordagem de Aichernig é exclusiva do paradigma de aprendizado de autômatos — ela depende da existência de um modelo formal MEF com estados, transições e saídas definidas. O Agilkia não produz um modelo formal, portanto não é compatível com esta técnica.

### Resultado esperado

O oráculo de Aichernig deve confirmar a corretude do modelo com significativamente menos queries do que o oráculo aleatório. Como o SUL é a própria Trie dos logs, não há divergência a encontrar — o modelo já reflete os dados perfeitamente. O benefício desta abordagem seria mais evidente com um sistema real em execução.

In [ ]:
def gerar_mutantes_mef(model):
    """
    Gera todos os mutantes da MEF por:
    - Output Fault: troca a saída de uma transição
    - Transfer Fault: redireciona uma transição para outro estado
    """
    mutantes = []
    estados  = model.states
    alfabeto = model.get_input_alphabet()

    for estado in estados:
        for simbolo in alfabeto:
            saida_orig   = estado.output_fun[simbolo]
            destino_orig = estado.transitions[simbolo]

            # Output Fault
            outras_saidas = list(set(['ok', 'erro', 'void', 'valor', 'nao_observado']) - {saida_orig})
            for nova_saida in outras_saidas:
                mutantes.append({
                    'tipo': 'output_fault', 'estado': estado.state_id,
                    'simbolo': simbolo, 'saida_orig': saida_orig,
                    'saida_mut': nova_saida, 'destino_orig': destino_orig.state_id,
                    'destino_mut': destino_orig.state_id,
                })

            # Transfer Fault
            outros_destinos = [s for s in estados if s.state_id != destino_orig.state_id]
            for novo_destino in outros_destinos:
                mutantes.append({
                    'tipo': 'transfer_fault', 'estado': estado.state_id,
                    'simbolo': simbolo, 'saida_orig': saida_orig,
                    'saida_mut': saida_orig, 'destino_orig': destino_orig.state_id,
                    'destino_mut': novo_destino.state_id,
                })

    print(f'Total de mutantes gerados : {len(mutantes)}')
    print(f'  Output faults           : {sum(1 for m in mutantes if m["tipo"] == "output_fault")}')
    print(f'  Transfer faults         : {sum(1 for m in mutantes if m["tipo"] == "transfer_fault")}')
    return mutantes

mutantes_mef = gerar_mutantes_mef(model_trie)

In [ ]:
def encontrar_sequencia_distinguidora(model, mutante, max_profundidade=10):
    """
    Busca BFS uma sequência que produz saída diferente
    entre o modelo original e o mutante.
    """
    from collections import deque
    alfabeto   = model.get_input_alphabet()
    estado_map = {s.state_id: s for s in model.states}
    estado_mut_id  = mutante['estado']
    simbolo_mut    = mutante['simbolo']
    saida_mut      = mutante['saida_mut']
    destino_mut_id = mutante['destino_mut']

    def step_original(estado_id, simbolo):
        s = estado_map[estado_id]
        return s.output_fun[simbolo], s.transitions[simbolo].state_id

    def step_mutante(estado_id, simbolo):
        if estado_id == estado_mut_id and simbolo == simbolo_mut:
            if mutante['tipo'] == 'output_fault':
                return saida_mut, estado_map[estado_id].transitions[simbolo].state_id
            else:
                return estado_map[estado_id].output_fun[simbolo], destino_mut_id
        s = estado_map[estado_id]
        return s.output_fun[simbolo], s.transitions[simbolo].state_id

    fila      = deque([([], model.initial_state.state_id, model.initial_state.state_id)])
    visitados = set()

    while fila:
        sequencia, est_orig, est_mut = fila.popleft()
        if len(sequencia) > max_profundidade: continue
        chave = (tuple(sequencia), est_orig, est_mut)
        if chave in visitados: continue
        visitados.add(chave)
        for simbolo in alfabeto:
            saida_o, prox_orig = step_original(est_orig, simbolo)
            saida_m, prox_mut  = step_mutante(est_mut,  simbolo)
            nova_seq = sequencia + [simbolo]
            if saida_o != saida_m:
                return nova_seq, saida_o, saida_m
            fila.append((nova_seq, prox_orig, prox_mut))
    return None, None, None

print('Gerando sequências distinguidoras...\n')
distinguidoras = []
for mutante in mutantes_mef:
    seq, saida_o, saida_m = encontrar_sequencia_distinguidora(model_trie, mutante)
    if seq:
        distinguidoras.append({'mutante': mutante, 'sequencia': seq,
                               'saida_orig': saida_o, 'saida_mut': saida_m})

print(f'Mutantes com sequência distinguidora: {len(distinguidoras)}/{len(mutantes_mef)}')

vistos = set()
contraexemplos_unicos = []
for d in distinguidoras:
    chave = tuple(d['sequencia'])
    if chave not in vistos:
        vistos.add(chave)
        contraexemplos_unicos.append(d['sequencia'])

print(f'Sequências únicas  : {len(contraexemplos_unicos)}')
print(f'Tamanhos: min={min(len(s) for s in contraexemplos_unicos)} '
      f'max={max(len(s) for s in contraexemplos_unicos)} '
      f'média={sum(len(s) for s in contraexemplos_unicos)/len(contraexemplos_unicos):.1f}')

In [ ]:
class AichernigOracle:
    """
    Oráculo de equivalência baseado em Aichernig & Tappler (2017, 2019).
    Fase 1: testa sequências distinguidoras geradas por mutação da MEF.
    Fase 2: fallback para busca aleatória se nenhum contraexemplo for encontrado.
    """
    def __init__(self, alphabet, sul, contraexemplos, num_random=500):
        self.alphabet       = alphabet
        self.sul            = sul
        self.contraexemplos = contraexemplos
        self.num_queries    = 0
        self.num_steps      = 0
        self.random_oracle  = RandomWpMethodEqOracle(
            alphabet, sul, min_length=2, expected_length=10, num_tests=num_random
        )

    def find_cex(self, hypothesis):
        for seq in self.contraexemplos:
            self.sul.pre()
            self.num_queries += 1
            self.num_steps   += len(seq)
            saidas_hyp, saidas_sul = [], []
            for simbolo in seq:
                saidas_hyp.append(hypothesis.step(simbolo))
                saidas_sul.append(self.sul.step(simbolo))
            hypothesis.reset_to_initial()
            self.sul.post()
            if saidas_hyp != saidas_sul:
                return seq
        cex = self.random_oracle.find_cex(hypothesis)
        self.num_queries += self.random_oracle.num_queries
        self.num_steps   += self.random_oracle.num_steps
        return cex

sul_aichernig    = LogSUL(traces_com_status)
oracle_aichernig = AichernigOracle(alphabet_real, sul_aichernig, contraexemplos_unicos)

model_aichernig = run_Lstar(
    alphabet_real, sul_aichernig, oracle_aichernig,
    automaton_type='mealy'
)

print(f'\nModelo aprendido com {len(model_aichernig.states)} estados.')
print(f'\n{"="*50}')
print(' COMPARATIVO DE ORÁCULOS')
print(f'{"="*50}')
print(f'{"Métrica":<35} {"Aleatório":>12} {"Aichernig":>12}')
print(f'{"-"*50}')
print(f'{"Estados aprendidos":<35} {"4":>12} {len(model_aichernig.states):>12}')
print(f'{"MQ — Equivalência":<35} {"2000":>12} {oracle_aichernig.num_queries:>12}')
reducao = (1 - oracle_aichernig.num_queries/2000) * 100
print(f'{"Redução de queries":<35} {"-":>12} {f"{reducao:.1f}%":>12}')
print(f'{"="*50}')

## 9. Cobertura Comparativa — AALpy vs Agilkia

Para uma comparação justa, aplicamos a **mesma lógica de cobertura** nos dois lados:

$$\text{Cobertura} = \frac{\text{comportamentos cobertos}}{\text{total de comportamentos possíveis}}$$

O denominador é o mesmo para os dois (36), mas a natureza do que é medido difere:

- **AALpy:** cobertura de transições — `(estado, evento)` visitados / total de transições da MEF (4 estados × 9 eventos = 36)
- **Agilkia:** cobertura de combinações — `(action, status)` observados / total de combinações possíveis (9 actions × 4 status = 36)

Ambas utilizam a mesma lógica — cobertura sobre o espaço de comportamentos possíveis — adaptada à natureza de cada abordagem. Essa diferença é justificável e deve ser explicitada na dissertação.

**Nota sobre o `nao_observado`:** as transições com saída `nao_observado` representam comportamentos que o modelo inventou para ser completo (estado lixo s3), não comportamentos reais. Por isso são excluídas da análise comparativa.

In [ ]:
from collections import deque

def gerar_sequencias_da_mef(model, max_profundidade=5):
    """Gera sequências de teste cobrindo todas as transições da MEF via BFS."""
    alfabeto  = model.get_input_alphabet()
    sequencias = []
    visitadas  = set()
    fila = deque([([], model.initial_state)])
    while fila:
        sequencia, estado = fila.popleft()
        if len(sequencia) > max_profundidade: continue
        for simbolo in alfabeto:
            chave = (estado.state_id, simbolo)
            nova_seq = sequencia + [simbolo]
            if chave not in visitadas:
                visitadas.add(chave)
                sequencias.append(nova_seq)
            prox_estado = estado.transitions[simbolo]
            fila.append((nova_seq, prox_estado))
    print(f'Sequências geradas pela MEF : {len(sequencias)}')
    print(f'Transições cobertas         : {len(visitadas)}/{len(model.states) * len(alfabeto)}')
    return sequencias

def executar_sequencias_no_sul(sequencias, sul):
    """Executa cada sequência no SUL e coleta saídas reais."""
    traces_gerados = []
    for seq in sequencias:
        sul.pre()
        trace = []
        for simbolo in seq:
            saida = sul.step(simbolo)
            trace.append((simbolo.strip(), saida))
        sul.post()
        traces_gerados.append(trace)
    return traces_gerados

sequencias_mef = gerar_sequencias_da_mef(model_trie)
sul_temp       = LogSUL(traces_com_status)
traces_mef_sul = executar_sequencias_no_sul(sequencias_mef, sul_temp)

# Filtra nao_observado dos dois lados
trace_list_mef_filtrado = []
for trace in traces_mef_sul:
    events = [agilkia.Event(action=ev, inputs={}, outputs={'status': saida})
              for ev, saida in trace if saida != 'nao_observado']
    if events:
        trace_list_mef_filtrado.append(agilkia.Trace(events))
trace_set_mef_filtrado = agilkia.TraceSet(trace_list_mef_filtrado)

trace_list_ag_filtrado = []
for trace in trace_set:
    events = [ev for ev in trace if ev.outputs.get('status','') != 'nao_observado']
    if events:
        trace_list_ag_filtrado.append(agilkia.Trace(events))
trace_set_ag_filtrado = agilkia.TraceSet(trace_list_ag_filtrado)

# Cobertura no mesmo universo
actions_unicas        = set(ev.action for t in trace_set for ev in t)
status_unicos         = set(ev.outputs.get('status','') for t in trace_set for ev in t)
total_pares_possiveis = len(actions_unicas) * len(status_unicos)

pares_aalpy2   = set(f"{ev.action}_{ev.outputs.get('status','')}" for t in trace_set_mef_filtrado for ev in t)
pares_agilkia2 = set(f"{ev.action}_{ev.outputs.get('status','')}" for t in trace_set_ag_filtrado for ev in t)

cobertura_agilkia = len(pares_agilkia2) / total_pares_possiveis

print('=' * 60)
print(' COBERTURA COMPARATIVA — AALpy vs Agilkia')
print(' (mesma lógica, mesmo denominador, naturezas adaptadas)')
print('=' * 60)
print(f'AALpy   : {cobertura_pct:.1f}%  ({len(transicoes_cobertas)}/{len(todas_transicoes)} transições)')
print(f'Agilkia : {cobertura_agilkia:.1%} ({len(pares_agilkia2)}/{total_pares_possiveis} combinações action×status)')
print()
so_agilkia = pares_agilkia2 - pares_aalpy2
print(f'Combinações só no Agilkia (MEF não capturou): {len(so_agilkia)}')
for p in sorted(so_agilkia):
    print(f'  ✘ {p}')
print('=' * 60)

## 10. Resultados e Visualizações

### Interpretação dos resultados

**Cobertura:** AALpy (38.9%) e Agilkia (36.1%) apresentam cobertura equivalente quando medidos pelo mesmo critério. A diferença de 2.8 pontos percentuais não é significativa. O que difere é a natureza — AALpy mede cobertura estrutural do modelo formal; Agilkia mede cobertura comportamental dos dados.

**Mutation Score:** o Agilkia detecta 100% dos mutantes por ser extremamente sensível a qualquer perturbação nos dados (compara conjuntos de pares diretamente). O AALpy detecta 58.9% porque aprende um modelo compacto que abstrai pequenas variações — menos sensível a ruído, mas formalmente verificável. RPNI e GSM detectam 0% por produzirem modelos degenerados de 1 estado.

**Aichernig:** confirmou a corretude do modelo com 71% menos queries que o oráculo aleatório (575 vs 2000), evidenciando a eficiência da abordagem baseada em mutação da MEF.

**Conclusão:** a escolha entre L* e Agilkia depende do objetivo. Se o foco é detecção sensível de qualquer perturbação, o Agilkia é mais eficaz. Se o foco é um modelo verificável formalmente e generalizável além dos dados, o L* é superior.

In [ ]:
operadores_nomes = ['output_fault', 'transfer_fault', 'remove_evento']
algoritmos = ['RPNI\n(Passivo)', 'GSM\n(Passivo)', 'L* AALpy\n(Ativo)', 'Agilkia\n(Estatístico)']

def extrair_scores(resultados):
    return [score for _, _, _, score in resultados]

scores = {
    'RPNI':    extrair_scores(resultados_rpni),
    'GSM':     extrair_scores(resultados_gsm),
    'L*':      extrair_scores(resultados_aalpy),
    'Agilkia': extrair_scores(resultados_agilkia),
}
totais = {k: sum(v)/len(v) for k, v in scores.items()}
cores  = {'RPNI': 'lightcoral', 'GSM': 'lightsalmon', 'L*': 'steelblue', 'Agilkia': 'darkorange'}
chaves = ['RPNI', 'GSM', 'L*', 'Agilkia']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(operadores_nomes))
width = 0.2
offsets = [-1.5, -0.5, 0.5, 1.5]
ax1 = axes[0]
for chave, offset in zip(chaves, offsets):
    bars = ax1.bar(x + offset * width, scores[chave], width,
                   label=chave, color=cores[chave], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, h + 1,
                     f'{h:.0f}%', ha='center', va='bottom', fontsize=8)
ax1.set_xlabel('Operador de Mutação', fontsize=12)
ax1.set_ylabel('Mutation Score (%)', fontsize=12)
ax1.set_title('Mutation Score por Operador\n(RPNI vs GSM vs L* vs Agilkia)', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(operadores_nomes, fontsize=10)
ax1.set_ylim(0, 120)
ax1.legend(fontsize=10)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2 = axes[1]
valores    = [totais[k] for k in chaves]
cores_plot = [cores[k] for k in chaves]
bars = ax2.bar(algoritmos, valores, color=cores_plot, edgecolor='black', width=0.5)
for bar, val in zip(bars, valores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mutation Score Médio (%)', fontsize=12)
ax2.set_title(f'Mutation Score Total por Algoritmo\n(n={N_MUTANTES} mutantes por operador)', fontsize=13)
ax2.set_ylim(0, 120)
ax2.axhline(y=50, color='gray', linestyle=':', linewidth=1.5, label='50% referência')
ax2.legend(fontsize=10)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('mutation_score_4algoritmos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: mutation_score_4algoritmos.png')

In [ ]:
ns, cobs_asc, cobs_epc = zip(*resultados_cob)

fig2, ax3 = plt.subplots(figsize=(10, 5))
ax3.plot(ns, [c*100 for c in cobs_asc], marker='o', label='Agilkia — Action+Status Coverage', color='darkorange')
ax3.plot(ns, [c*100 for c in cobs_epc], marker='s', label='Agilkia — EventPair Coverage', color='gold', linestyle='--')
ax3.axhline(y=cobertura_pct, color='steelblue', linestyle='-.', linewidth=1.5,
            label=f'AALpy — Cobertura de Transições ({cobertura_pct:.1f}%)')
ax3.set_xlabel('Número de traces selecionados', fontsize=12)
ax3.set_ylabel('Cobertura (%)', fontsize=12)
ax3.set_title('Cobertura por Tamanho do Subconjunto\nAgilkia vs AALpy (referência)', fontsize=13)
ax3.set_xscale('log')
ax3.set_ylim(0, 110)
ax3.legend(fontsize=10)
ax3.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('cobertura_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: cobertura_comparativa.png')

In [ ]:
r_rpni        = f"{totais['RPNI']:.1f}%"
r_gsm         = f"{totais['GSM']:.1f}%"
r_lstar       = f"{totais['L*']:.1f}%"
r_agil        = f"{totais['Agilkia']:.1f}%"
n_states      = str(len(model_trie.states))
cob_aalpy_str = f"{cobertura_pct:.1f}%"
cob_agil_str  = f"{cobertura_agilkia:.1%}"

print('\n' + '=' * 70)
print(' TABELA COMPARATIVA FINAL — 4 Algoritmos')
print('=' * 70)
print(f'{"Métrica":<35} {"RPNI":<10} {"GSM":<10} {"L*":<12} {"Agilkia"}')
print('-' * 70)
print(f'{"Estados no modelo":<35} {"1":<10} {"1":<10} {n_states:<12} {"N/A"}')
print(f'{"Cobertura (mesmo critério)":<35} {"N/A":<10} {"N/A":<10} {cob_aalpy_str:<12} {cob_agil_str}')
print(f'{"  AALpy: trans. cobertas/total":<35} {"N/A":<10} {"N/A":<10} {"14/36":<12} {"N/A"}')
print(f'{"  Agilkia: pares cobertos/total":<35} {"N/A":<10} {"N/A":<10} {"N/A":<12} {"13/36"}')

labels = ['output_fault', 'transfer_fault', 'remove_evento']
for i, label in enumerate(labels):
    r  = f"{scores['RPNI'][i]:.1f}%"
    g  = f"{scores['GSM'][i]:.1f}%"
    ls = f"{scores['L*'][i]:.1f}%"
    ag = f"{scores['Agilkia'][i]:.1f}%"
    print(f'{"Mut. Score — " + label:<35} {r:<10} {g:<10} {ls:<12} {ag}')

print(f'{"Mut. Score TOTAL":<35} {r_rpni:<10} {r_gsm:<10} {r_lstar:<12} {r_agil}')
print(f'{"Modelo formal (MEF)":<35} {"Sim":<10} {"Sim":<10} {"Sim":<12} {"Não"}')
print(f'{"Verificação formal":<35} {"Limitada":<10} {"Limitada":<10} {"Sim":<12} {"Não"}')
print('=' * 70)